## Create a SQL Metric View to Calculate Total Emission Values

### Create the Metric View in Unity Catalog

In [ ]:
%sql
  
CREATE OR REPLACE VIEW carbonops.gold.emissions_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1

comment: >
  CarbonOps governed emissions metrics for
  sustainability analytics and AI/BI Genie.

source: carbonops.gold.facility_emissions

fields:
  - name: Facility
    expr: source.facility_name
    comment: Name of the manufacturing facility

  - name: City
    expr: source.city
    comment: City where the facility operates

  - name: State
    expr: source.state
    comment: State where the facility operates

  - name: Industry
    expr: source.industry
    comment: Industry classification of the facility

  - name: Year
    expr: source.year
    comment: Reporting year

  - name: Month
    expr: source.month
    comment: Reporting month number

measures:
  - name: Total Emissions
    expr: SUM(source.total_tco2e)
    comment: Total Scope 1 and Scope 2 emissions in tCO2e

  - name: Scope 1 Emissions
    expr: SUM(source.scope1_tco2e)
    comment: Direct emissions in tCO2e

  - name: Scope 2 Emissions
    expr: SUM(source.scope2_tco2e)
    comment: Indirect electricity emissions in tCO2e

  - name: Total Electricity
    expr: SUM(source.total_electricity_kwh)
    comment: Total purchased electricity in kWh

  - name: Average Monthly Emissions
    expr: AVG(source.total_tco2e)
    comment: Average emissions per facility-month record

  - name: Facility Count
    expr: COUNT(DISTINCT source.facility_id)
    comment: Number of distinct reporting facilities
$$;

### Query the Metric View

In [ ]:
%sql

SELECT
    `City`,
    MEASURE(`Total Emissions`) AS total_emissions,
    MEASURE(`Scope 1 Emissions`) AS scope1_emissions,
    MEASURE(`Scope 2 Emissions`) AS scope2_emissions
FROM carbonops.gold.emissions_metrics
GROUP BY `City`
ORDER BY total_emissions DESC;

### Validate Yearly Metrics

In [ ]:
%sql 

SELECT
    `Year`,
    MEASURE(`Total Emissions`) AS total_emissions,
    MEASURE(`Total Electricity`) AS electricity_kwh,
    MEASURE(`Facility Count`) AS facility_count
FROM carbonops.gold.emissions_metrics
GROUP BY `Year`
ORDER BY `Year`;